<!--
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0
-->


# Lab 0: Environment Setup

## What this notebook does, and why it exists

Nothing in this notebook trains anything. Its entire job is to **discover the environment CloudFormation already built for you** and write down the handful of identifiers that Labs 1-4 will need — so you configure them once here instead of four times later.

By the end of this notebook you will have:

| What | Why the rest of the workshop needs it |
|---|---|
| SageMaker session, account ID, region | Every training and evaluation job is submitted through these |
| SageMaker execution role | The identity your training jobs assume to read S3 and write model artifacts |
| Lambda execution role | The identity the SQL evaluator you build in Lab 2 will run as |
| Default S3 bucket | Where datasets and model weights land |
| **An MLflow tracking server** | Every training job and evaluation in Labs 2-4 logs its metrics here automatically. By the end of the workshop it holds 11 experiments, and it is how you compare models |
| **Aurora cluster + secret ARNs** | The database your training data comes from |

All of it is persisted with `%store`, IPython's cross-notebook variable store. Each later lab starts with `%store -r` and picks these up — which is why **Lab 0 must be run first, and re-run if your kernel restarts.**

## Where the workshop is going

```
  Lab 0  discover environment, create MLflow          ← you are here
    │
  Lab 1  extract real queries from the database,
         turn them into a training dataset with Bedrock
    │
  Lab 2  build a SQL evaluator (Lambda), fine-tune
         Llama 3.2 3B on your data with SFT, score it
    │
  Lab 3  reinforcement learning on top of SFT, using
         that same evaluator as the reward function
         + an ablation: RLVR without SFT first
    │
  Lab 4  benchmark everything against frontier models
```

The one thing worth internalizing early: **the evaluator you build in Lab 2 does double duty.** It is the metric that scores SFT in Lab 2, and it is the reward function that trains the model in Lab 3. That is what makes reinforcement learning possible here at all — SQL correctness can be checked by *running the query*, so the reward is just code, not a human labelling exercise.

## Install Dependencies

This workshop uses the SageMaker Python SDK v3 for the `SFTTrainer`, `RLVRTrainer`, and `DataSet` APIs. The versions below are pinned to the exact set the labs were validated against — the SDK is under active development so please don't change the pinned versions.

Restart the kernel after this cell completes.

In [ ]:
# Pinned to the exact versions this workshop was validated against.
# The `sagemaker` meta-package only range-depends on its subpackages
# (sagemaker-train<2.0.0,>=1.18.0 etc.), and the trainer APIs used in Labs 2-4
# live in sagemaker-train — so every subpackage is pinned explicitly. Do not
# relax these to --upgrade: minor releases have changed trainer behavior.
!pip install --quiet \
    sagemaker==3.18.0 \
    sagemaker-core==2.18.0 \
    sagemaker-train==1.18.0 \
    sagemaker-serve==1.18.0 \
    sagemaker-mlops==1.18.0

# restart the kernel after running this cell
import importlib.metadata as md
for p in ("sagemaker", "sagemaker-core", "sagemaker-train", "sagemaker-serve", "sagemaker-mlops"):
    print(f"{p:18s} {md.version(p)}")

<h1 style="color: #d32f2f; text-align: center;">DO NOT PROCEED WITHOUT RESTARTING THE KERNEL</h1>

<p style="text-align: center; color: #666;">On the menu bar, click <b>Kernel</b> → <b>Restart Kernel...</b> then continue to the next cell.</p>

## Configure Session and MLflow

This cell sets up the SageMaker session, identifies the execution role and S3 bucket, and creates (or finds) an MLflow tracking server. MLflow is where all training metrics and evaluation results are logged — it gives us a single dashboard to compare models across labs.

In [ ]:
# Setup SageMaker session
import boto3
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.resources import MlflowApp

# Get Account ID
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)

# Create SageMaker session
sagemaker_session = Session(sagemaker_client=sm_client)

ROLE = get_execution_role()

LAMBDA_ROLE = f"arn:aws:iam::{ACCOUNT_ID}:role/lambda-evaluator-execution-role"

DEFAULT_BUCKET = sagemaker_session.default_bucket()

print(f"Account Id: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"default bucket: {DEFAULT_BUCKET}")
print(f"SageMaker Execution Role: {ROLE}")
print(f"Lambda Execution Role: {LAMBDA_ROLE}")


# Get MLFlow ARN
# Name of the MLflow app created for this workshop. It is stored below so the
# clean-up notebook can find and delete exactly this app (no hard-coded names).
mlflow_app_name = "workshop-mlflow"

mlflow_app = None
try:
    apps = list(MlflowApp.get_all())
    for app in apps:
        if app.status not in ("Created", "Updated"):
            continue
        if app.name == mlflow_app_name:
            mlflow_app = app
            print(f"Found existing MLflow app: {mlflow_app.name}")
            break
except Exception as e:
    print(f"No matching MLflow app found: {e}")

if mlflow_app is None:
    print(f"Creating new MLflow app: {mlflow_app_name}...")
    mlflow_app = MlflowApp.create(
        name=mlflow_app_name,
        artifact_store_uri=f"s3://{DEFAULT_BUCKET}/mlflow",
        role_arn=ROLE,
        model_registration_mode="AutoModelRegistrationEnabled",
    )
    print(f"Creating MLflow app: {mlflow_app.name}")


MLFLOW_ARN = mlflow_app.arn

print(f"MLFlow ARN: {MLFLOW_ARN}")

## Locate the Aurora Database

### Why is there a PostgreSQL database in an LLM workshop?

Because that is where the training data comes from.

Most text-to-SQL tutorials hand you a curated dataset of question/query pairs. This workshop doesn't. Instead, CloudFormation deployed an Aurora PostgreSQL Serverless v2 cluster, seeded it with a `product_sales` table from a cosmetics sales dataset, and then **replayed roughly 250 realistic analytical queries against it** — filters, aggregations, window functions, date ranges.

PostgreSQL recorded every one of them in `pg_stat_statements`, an extension that tracks aggregated execution statistics for every statement the server runs. That view is your training corpus. In Lab 1 you will read it, and every query you fine-tune on will be a query that genuinely ran against this schema — with real call counts and real execution times attached.

This matters more than it sounds. It is the difference between the workshop demonstrating a technique and the workshop demonstrating a *workflow*: on a real project, your query log is exactly where you would start, and it is almost always more representative of what users ask than anything you would write by hand.

### The three values below

| Variable | What it is | Used by |
|---|---|---|
| `AURORA_CLUSTER_ARN` | The cluster, addressed through the RDS **Data API** — HTTP SQL, no VPC connection or driver needed from the notebook | Lab 1 (extract queries), Lab 2 evaluator Lambda |
| `AURORA_SECRET_ARN` | Secrets Manager secret holding the DB credentials. Nothing in this workshop ever handles a password directly | Same |
| `AURORA_DB_NAME` | The database name | Same |

Together these are how both the notebook and the evaluator Lambda execute SQL: pass all three to `rds-data`, get results back as JSON. That is also what makes the Lab 2 reward function possible — scoring generated SQL means *actually running it*, and the Data API makes that a single API call.

In [ ]:
# Aurora cluster details, reconstructed from the CloudFormation stack's naming convention.
#
# The "dev-" prefix and "dev/" path mirror the stack's EnvironmentName parameter,
# which defaults to "dev". If you redeploy sm-stack.yaml with a different
# EnvironmentName, these two lines must be updated to match.
#
# Note the secret ARN deliberately omits the random 6-character suffix that
# Secrets Manager appends. Both Secrets Manager and the RDS Data API accept a
# partial ARN, so this resolves correctly as written.
AURORA_CLUSTER_ARN = f"arn:aws:rds:{REGION}:{ACCOUNT_ID}:cluster:dev-query-training-cluster"
AURORA_SECRET_ARN = f"arn:aws:secretsmanager:{REGION}:{ACCOUNT_ID}:secret:dev/rds/query-training-db"
AURORA_DB_NAME = "querytraining"

print(f"Aurora Cluster: {AURORA_CLUSTER_ARN}")
print(f"Aurora Secret:  {AURORA_SECRET_ARN}")
print(f"Aurora DB:      {AURORA_DB_NAME}")

# Persist key variables for use in subsequent lab notebooks.
# Every later notebook begins with `%store -r` to read these back, so if your
# kernel restarts mid-workshop, re-run this notebook rather than the later one.
%store ACCOUNT_ID REGION ROLE LAMBDA_ROLE DEFAULT_BUCKET MLFLOW_ARN
%store AURORA_CLUSTER_ARN AURORA_SECRET_ARN AURORA_DB_NAME

### Verify the database is reachable

The three ARNs above are *reconstructed* from the stack's naming convention, not read back from CloudFormation — so a wrong `EnvironmentName` would leave them syntactically valid but pointing at nothing. The cell below catches that **here, in Lab 0**, rather than letting it surface as a confusing failure in Lab 1.

It runs a trivial `SELECT 1` through the RDS Data API, which exercises all three values at once: the cluster ARN, the secret ARN, and the database name. If it prints a success line you are good for the rest of the workshop; if it fails, the output tells you what to change.

In [ ]:
# Confirm the reconstructed ARNs actually resolve, by running SELECT 1 through
# the Data API. This validates the cluster ARN, the secret ARN and the database
# name together -- if any is wrong, this is where you find out.
rds_data = boto3.client("rds-data", region_name=REGION)

try:
    rds_data.execute_statement(
        resourceArn=AURORA_CLUSTER_ARN,
        secretArn=AURORA_SECRET_ARN,
        database=AURORA_DB_NAME,
        sql="SELECT 1",
    )
    print("Database reachable — Aurora ARNs and secret verified. You are ready for Lab 1.")
except Exception as e:
    name = type(e).__name__
    print(f"Could not reach the database: {name}\n{e}\n")
    print("Troubleshooting:")
    print("  - This almost always means the stack was deployed with a different")
    print("    EnvironmentName than the default 'dev'. Check the sm-stack")
    print("    CloudFormation stack's EnvironmentName parameter, then update the")
    print("    'dev-' prefix and 'dev/' path in the cell above to match and re-run.")
    print("  - A DatabaseResumingException/timeout can mean Aurora Serverless v2")
    print("    is still settling just after the stack finished deploying. Wait")
    print("    ~60s and re-run this cell; if it still fails after two or three")
    print("    tries, the cause is almost certainly the EnvironmentName above,")
    print("    not the database. (This cluster idles at 0.5 ACU and does not")
    print("    scale to zero, so there is no long cold start to wait out.)")
    print("  - Confirm the sm-stack CloudFormation stack finished with status")
    print("    CREATE_COMPLETE.")
    raise

### Verify Bedrock model access

Lab 1 calls **Claude Sonnet** on Amazon Bedrock to turn each SQL query into a natural-language question, and Lab 4 benchmarks against Claude on Bedrock too. In an AWS event account this access is pre-configured — but confirming it now, with a one-token test call, beats discovering a missing grant ten minutes into Lab 1's cleaning loop.

The cell below invokes the exact model Lab 1 uses (`us.anthropic.claude-sonnet-4-6`) with a trivial prompt. A success line means you are ready; a failure tells you what to check.

In [ ]:
import json

# Verify Bedrock access by invoking the model Lab 1 actually uses, with a
# 1-token prompt. This exercises the real path (bedrock-runtime:InvokeModel on
# the Claude inference profile) rather than just listing models -- listing can
# succeed while invocation is still denied.
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-6"
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

try:
    bedrock_runtime.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1,
            "messages": [{"role": "user", "content": "ping"}],
        }),
    )
    print(f"Bedrock reachable — invoked {BEDROCK_MODEL_ID} successfully. You are ready for Lab 1.")
except Exception as e:
    name = type(e).__name__
    print(f"Could not invoke Bedrock model {BEDROCK_MODEL_ID}: {name}\n{e}\n")
    print("Troubleshooting:")
    print("  - AccessDeniedException: model access is not enabled in this account/region.")
    print("    At an AWS event this should be pre-configured -- confirm you are in the")
    print("    event's region (shown above) and have not switched regions. Outside an")
    print("    event, enable Claude model access in the Bedrock console under")
    print("    'Model access'.")
    print("  - ValidationException on the modelId: the cross-region inference profile")
    print("    may differ in your region. Check the exact profile ID in the Bedrock")
    print("    console under 'Cross-region inference'.")
    print("  - ThrottlingException: transient -- wait a moment and re-run this cell.")
    raise

## Where to find MLflow

You just created a tracking server, and from Lab 2 onwards every training job and every evaluation writes to it automatically. There are two ways to open it:

1. **From Studio (easiest)** — in the SageMaker Studio left sidebar, under **Applications**, click the **MLflow** tile. Same panel you launched JupyterLab from.
2. **From the AWS Console** — Amazon SageMaker AI → **MLflow** under *Applications and IDEs* → open `workshop-mlflow`.

Open it now and confirm it loads. It will be empty until Lab 2 — that is expected. The cell below prints a direct console link.

There is a full walkthrough of what to look at in each experiment in the **Reviewing Results** module at the end of the workshop.

In [ ]:
# Print a clickable link to the MLflow UI.
#
# CreatePresignedMlflowAppUrl mints a short-lived signed URL that logs you
# straight into the tracking server. If that API isn't available in this
# environment's boto3, we fall back to the console landing page.
from IPython.display import display, HTML


def mlflow_link():
    url = None
    try:
        url = sm_client.create_presigned_mlflow_app_url(Arn=MLFLOW_ARN)["AuthorizedUrl"]
        label = "Open MLflow (signed link, valid ~5 minutes)"
    except Exception as e:
        print(f"Presigned URL unavailable ({type(e).__name__}), falling back to console link.")
        url = f"https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/mlflow"
        label = "Open MLflow in the AWS Console"
    display(HTML(f'<a href="{url}" target="_blank"><b>{label}</b></a>'))
    print(f"\nTracking server: {mlflow_app_name}")
    print(f"Tracking URI:    {MLFLOW_ARN}")


mlflow_link()